In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
#  REACTOR PRESSURE ANALYZER  —  Cell 1: Imports, Config & Functions
#  Run this cell once, then run Cell 2 to display the GUI.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Imports ──────────────────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import threading
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.ticker import AutoMinorLocator
import numpy as np
import pandas as pd

# ── Step styling ──────────────────────────────────────────────────────────────
step_dict = {
    "Wait Before Start": {'color': '#dab1da', 'outline': '#986998', 'marker': 's'},
    "Predose Pump":      {'color': '#f0d3ef', 'outline': '#a881a7', 'marker': 'o'},
    "N2Overlap":         {'color': '#c9d9ec', 'outline': '#8b96a4', 'marker': 's'},
    "PumpOverlap":       {'color': '#dceaf7', 'outline': '#9aa3ac', 'marker': 'o'},
    "Open Time":         {'color': '#b9e0c6', 'outline': '#7f9c88', 'marker': 's'},
    "Hold":              {'color': '#d5edd8', 'outline': '#95a59a', 'marker': 'o'},
    "Pump Purge":        {'color': '#fbd2a5', 'outline': '#b28774', 'marker': 's'},
    "Purge":             {'color': '#ffe0cb', 'outline': '#b29c8e', 'marker': 'o'},
}

# ── Global data (populated on file load) ──────────────────────────────────────
header_info     = {}
pressure_df     = pd.DataFrame()   # columns: time_ms, pressure, cycle
step_df         = pd.DataFrame()   # columns: Cycle, Step, Start/End Time (ms), Duration, Max/Steady Pressure, Slope
_step_times_g   = []               # N+1 floats: transition time for each step + final time
_step_labels_g  = []               # N strings
_unique_steps_g = []

# ── Steady-state parameters (mutable dict so callbacks update in-place) ───────
ss = dict(frac=0.1, slope_tol=15.0, amp_tol=100.0, min_pts=5)


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                            FILE LOADING                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def browse_file(b=None):
    """Open a native file dialog and populate file_path_text (requires local kernel)."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        path = filedialog.askopenfilename(
            title="Select Reactor Data JSON",
            filetypes=[("JSON files", "*.json"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            file_path_text.value = str(Path(path))
    except Exception as e:
        load_status.value = f"Browse unavailable ({e}). Paste path manually."


def _stream_parse(filepath: Path, prog_bar, status_lbl):
    """
    Stream-parse the newline-delimited JSON file.
    Only 'pressure' and 'step' records are stored — header is extracted once.
    Memory usage stays flat regardless of file size.
    """
    global header_info, pressure_df, step_df, _step_times_g, _step_labels_g, _unique_steps_g

    file_size  = max(filepath.stat().st_size, 1)
    p_times, p_values, p_cycles = [], [], []
    s_times, s_names, s_cycles  = [], [], []
    last_step  = None
    h_info     = {}
    bytes_read = 0

    with filepath.open('r', encoding='utf-8') as fh:
        for line in fh:
            bytes_read += len(line.encode())
            try:
                obj = json.loads(line)
                t   = obj.get('type')
                pl  = obj.get('payload', {})
                if isinstance(pl, str):
                    pl = json.loads(pl)

                if t == 'header':
                    h_info = pl

                elif t == 'pressure':
                    p_times.append(float(pl.get('TimeElapsed', 0)))
                    p_values.append(float(pl.get('Pressure', float('nan'))))
                    p_cycles.append(int(pl.get('CurrentCycle', 0)))

                elif t == 'step':
                    name = pl.get('CurrentStep', '')
                    if name != last_step:            # only record actual transitions
                        s_times.append(float(pl.get('TimeElapsed', 0)))
                        s_names.append(name)
                        s_cycles.append(int(pl.get('CurrentCycle', 0)))
                        last_step = name

            except Exception:
                pass

            if bytes_read % 3_000_000 < 3000:
                prog_bar.value = int(bytes_read / file_size * 80)

    # ── Build pressure DataFrame ──────────────────────────────────────────────
    pressure_df = pd.DataFrame({
        'time_ms':  np.array(p_times,  dtype=float),
        'pressure': np.array(p_values, dtype=float),
        'cycle':    np.array(p_cycles, dtype=int),
    }).sort_values('time_ms').reset_index(drop=True)

    max_time = float(pressure_df['time_ms'].max()) if not pressure_df.empty else 0.0

    # ── Build step_df with Max Pressure using searchsorted (no boolean masks) ─
    prog_bar.value = 82
    s_starts  = np.array(s_times, dtype=float)
    s_ends    = np.append(s_starts[1:], max_time)
    sorted_t  = pressure_df['time_ms'].to_numpy()
    sorted_p  = pressure_df['pressure'].to_numpy()

    max_pressures = []
    for lo_t, hi_t in zip(s_starts, s_ends):
        lo = int(np.searchsorted(sorted_t, lo_t, side='left'))
        hi = int(np.searchsorted(sorted_t, hi_t, side='right'))
        win = sorted_p[lo:hi]
        max_pressures.append(float(np.nanmax(win)) if len(win) else float('nan'))

    step_df = pd.DataFrame({
        'Cycle':           np.array(s_cycles, dtype=int),
        'Step':            s_names,
        'Start Time (ms)': s_starts,
        'End Time (ms)':   s_ends,
    })
    step_df['Duration (ms)']        = step_df['End Time (ms)'] - step_df['Start Time (ms)']
    step_df['Duration (s)']         = step_df['Duration (ms)'] / 1000.0
    step_df['Max Pressure (mTorr)'] = max_pressures
    step_df.reset_index(drop=True, inplace=True)

    # ── Update globals ────────────────────────────────────────────────────────
    header_info     = h_info
    _step_times_g   = step_df['Start Time (ms)'].tolist() + [max_time]
    _step_labels_g  = step_df['Step'].tolist()
    _unique_steps_g = list(dict.fromkeys(_step_labels_g))

    # ── Compute steady-state, then refresh UI ─────────────────────────────────
    prog_bar.value = 90
    _compute_steady()
    prog_bar.value = 100

    n_cyc = int(pressure_df[pressure_df['cycle'] > 0]['cycle'].max()) if not pressure_df.empty else 0
    status_lbl.value = (
        f"\u2714 {filepath.name}  |  "
        f"{len(pressure_df):,} pressure points  |  "
        f"{n_cyc} cycles"
    )
    _update_all_tabs()


def _load_worker(path_str, prog_bar, status_lbl):
    try:
        _stream_parse(Path(path_str), prog_bar, status_lbl)
    except Exception as exc:
        status_lbl.value = f"\u2716 Error: {exc}"
        prog_bar.value = 0


def load_file(b=None):
    path_str = file_path_text.value.strip()
    if not path_str:
        load_status.value = "Enter or browse for a file path."
        return
    p = Path(path_str)
    if not p.exists():
        load_status.value = f"\u2716 File not found: {p}"
        return
    load_status.value = "Loading\u2026"
    load_progress.value = 0
    threading.Thread(
        target=_load_worker, args=(path_str, load_progress, load_status), daemon=True
    ).start()


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                         STEADY-STATE ANALYSIS                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _steady_for_window(times_ms, pressures):
    valid = np.isfinite(times_ms) & np.isfinite(pressures)
    if not np.any(valid):
        return np.nan, np.nan
    t = np.asarray(times_ms[valid], dtype=float)
    p = np.asarray(pressures[valid], dtype=float)
    if t.size < 2:
        return np.nan, np.nan
    order  = np.argsort(t)
    t, p   = t[order], p[order]
    uniq_t, inv = np.unique(t, return_inverse=True)
    p = np.bincount(inv, weights=p) / np.bincount(inv)
    t = uniq_t
    times_s = t / 1000.0
    n       = t.size
    start   = max(int(np.floor(n * (1 - ss['frac']))), 0)
    st, sp  = times_s[start:], p[start:]
    if st.size < ss['min_pts']:
        return np.nan, np.nan
    dur = st[-1] - st[0]
    if dur <= 0:
        return np.nan, np.nan
    slope = np.polyfit(st, sp, 1)[0] if st.size >= 3 else (sp[-1] - sp[0]) / dur
    amp   = float(np.nanmax(sp) - np.nanmin(sp))
    steady = (abs(slope) <= ss['slope_tol']) and (amp <= ss['amp_tol'])
    return (float(np.nanmean(sp)) if steady else np.nan, float(slope))


def _compute_steady():
    if step_df.empty or pressure_df.empty:
        return
    sorted_t  = pressure_df['time_ms'].to_numpy()
    sorted_p  = pressure_df['pressure'].to_numpy()
    s_press, slopes = [], []
    for _, row in step_df.iterrows():
        lo = int(np.searchsorted(sorted_t, row['Start Time (ms)'], side='left'))
        hi = int(np.searchsorted(sorted_t, row['End Time (ms)'],   side='right'))
        sp, sl = _steady_for_window(sorted_t[lo:hi], sorted_p[lo:hi])
        s_press.append(sp)
        slopes.append(sl)
    step_df['Steady Pressure (mTorr)'] = s_press
    step_df['Slope (mTorr/s)']         = slopes


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                           PLOT FUNCTIONS                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _add_step_patches(ax, t_lo, t_hi):
    """Shade the axis background by step color within [t_lo, t_hi]."""
    if not _step_labels_g:
        return
    ylim = ax.get_ylim()
    for i, label in enumerate(_step_labels_g):
        s0, s1 = _step_times_g[i], _step_times_g[i + 1]
        if s1 <= t_lo or s0 >= t_hi:
            continue
        color = step_dict.get(label, {}).get('color', '#dddddd')
        ax.add_patch(mpatches.Rectangle(
            (max(s0, t_lo), ylim[0]),
            min(s1, t_hi) - max(s0, t_lo),
            ylim[1] - ylim[0],
            color=color, alpha=1.0, zorder=0,
        ))


def _add_step_legend(ax):
    present = [s for s in _unique_steps_g if s in step_dict]
    handles = [mpatches.Patch(color=step_dict[s]['color'], label=s) for s in present]
    if handles:
        ax.legend(handles=handles, title='Step',
                  bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)


def plot_pressure(cycle=None):
    if pressure_df.empty:
        print("No data loaded.")
        return

    if cycle is None:
        mask  = pressure_df['cycle'] > 0
        sub   = pressure_df[mask]
        t_lo  = sub['time_ms'].min() if not sub.empty else 0
        t_hi  = pressure_df['time_ms'].max()
        title = "Pressure vs Time Elapsed — All Cycles"
    else:
        cycle = int(cycle)
        sub   = pressure_df[pressure_df['cycle'] == cycle]
        t_lo  = sub['time_ms'].min() if not sub.empty else 0
        nxt   = pressure_df[pressure_df['cycle'] == cycle + 1]['time_ms'].min()
        t_hi  = float(nxt) if not pd.isna(nxt) else pressure_df['time_ms'].max()
        title = f"Pressure vs Time Elapsed — Cycle {cycle}"

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(sub['time_ms'], sub['pressure'], color='black', linewidth=1.5, zorder=2)
    _add_step_patches(ax, t_lo, t_hi)

    if cycle is None:
        c_starts = pressure_df[pressure_df['cycle'] > 0].groupby('cycle')['time_ms'].min()
        for c_t in c_starts:
            ax.axvline(c_t, color='red', linestyle='--', linewidth=1.2, alpha=0.6, zorder=3)

    ax.set_xlim(t_lo, t_hi)
    ax.set_title(title)
    ax.set_xlabel("Time Elapsed (ms)")
    ax.set_ylabel("Pressure (mTorr)")
    _add_step_legend(ax)
    plt.tight_layout()
    plt.rcParams['figure.dpi'] = 150
    plt.show()


def update_pressure_overlay():
    with pressure_overlay_output:
        clear_output(wait=True)
        if pressure_df.empty:
            return
        cycles = sorted(pressure_df[pressure_df['cycle'] > 0]['cycle'].unique())
        if len(cycles) < 2:
            return
        fig, ax = plt.subplots(figsize=(15, 5))
        cmap = cm.jet
        norm = mcolors.Normalize(vmin=int(cycles[0]), vmax=int(cycles[-1]))
        for c in cycles:
            sub = pressure_df[pressure_df['cycle'] == c]
            t0  = sub['time_ms'].min()
            ax.plot(sub['time_ms'] - t0, sub['pressure'],
                    linewidth=1.2, color=cmap(norm(c)))
        ax.set_xlim(0)
        ax.set_xlabel("Time Relative to Cycle Start (ms)")
        ax.set_ylabel("Pressure (mTorr)")
        ax.set_title("Overlay of All Cycles")
        ax.grid(which='major', linestyle='-',  linewidth=0.8, color='gray')
        ax.grid(which='minor', linestyle='--', linewidth=0.5, color='lightgray')
        ax.minorticks_on()
        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax).set_label('Cycle Number')
        plt.tight_layout()
        plt.show()


def plot_steady_state_patch(change=None):
    with steady_state_patch_output:
        clear_output(wait=True)
        if pressure_df.empty or step_df.empty:
            return
        cycle    = steady_cycle_dropdown.value
        selected = steady_step_selector.value

        if cycle == 'All':
            p_sub = pressure_df[pressure_df['cycle'] > 0]
            s_sub = step_df[step_df['Cycle'] > 0].copy()
        else:
            c     = int(cycle)
            p_sub = pressure_df[pressure_df['cycle'] == c]
            s_sub = step_df[step_df['Cycle'] == c].copy()

        if 'All' not in selected:
            s_sub = s_sub[s_sub['Step'].isin(selected)]

        fig, ax = plt.subplots(figsize=(15, 5))
        ax.plot(p_sub['time_ms'], p_sub['pressure'], color='black', linewidth=1.5)
        ylim = ax.get_ylim()
        for _, row in s_sub.iterrows():
            color = 'green' if not np.isnan(row['Steady Pressure (mTorr)']) else 'red'
            ax.add_patch(mpatches.Rectangle(
                (row['Start Time (ms)'], ylim[0]),
                row['End Time (ms)'] - row['Start Time (ms)'],
                ylim[1] - ylim[0],
                color=color, alpha=0.3, zorder=0,
            ))
        t_lo = p_sub['time_ms'].min() if not p_sub.empty else 0
        t_hi = p_sub['time_ms'].max() if not p_sub.empty else 1
        ax.set_xlim(t_lo, t_hi)
        ax.set_xlabel("Time Elapsed (ms)")
        ax.set_ylabel("Pressure (mTorr)")
        ax.set_title(f"Steady-State Classification — Cycle {cycle}")
        ax.legend(handles=[
            mpatches.Patch(color='green', alpha=0.3, label='Steady'),
            mpatches.Patch(color='red',   alpha=0.3, label='Not Steady'),
        ], bbox_to_anchor=(1.01, 1), loc='upper left')
        plt.tight_layout()
        plt.show()


def update_max_pressure(change=None):
    with max_pressure_output:
        clear_output(wait=True)
        if step_df.empty:
            print("No data loaded.")
            return
        selected = max_step_selector.value
        df_show  = step_df if 'All' in selected else step_df[step_df['Step'].isin(selected)]
        fig, ax  = plt.subplots(figsize=(6, 6))
        for step in df_show['Step'].unique():
            sub = df_show[df_show['Step'] == step]
            ax.scatter(sub['Cycle'], sub['Max Pressure (mTorr)'],
                       color=step_dict[step]['color'], edgecolor=step_dict[step]['outline'],
                       linewidth=0.7, marker=step_dict[step]['marker'], label=step)
        ax.set_title('Max Pressure per Cycle by Step')
        ax.set_xlabel('Cycle')
        ax.set_ylabel('Max Pressure (mTorr)')
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        c_min, c_max = int(step_df['Cycle'].min()), int(step_df['Cycle'].max())
        ax.set_xlim(c_min, c_max)
        ax.set_xticks(range(c_min, c_max + 1))
        ax.legend(title='Step', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
        display(df_show)


def update_steady_state_scatter(change=None):
    with steady_state_output:
        clear_output(wait=True)
        if step_df.empty:
            print("No data loaded.")
            return
        selected = steady_step_selector.value
        df_show  = step_df if 'All' in selected else step_df[step_df['Step'].isin(selected)]
        fig, ax  = plt.subplots(figsize=(6, 6))
        for step in df_show['Step'].unique():
            sub = df_show[df_show['Step'] == step]
            ax.scatter(sub['Cycle'], sub['Steady Pressure (mTorr)'],
                       color=step_dict[step]['color'], edgecolor=step_dict[step]['outline'],
                       linewidth=0.7, marker=step_dict[step]['marker'], label=step)
        ax.set_title('Steady-State Pressure per Cycle by Step')
        ax.set_xlabel('Cycle')
        ax.set_ylabel('Steady Pressure (mTorr)')
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        c_min, c_max = int(step_df['Cycle'].min()), int(step_df['Cycle'].max())
        ax.set_xlim(c_min, c_max)
        ax.set_xticks(range(c_min, c_max + 1))
        ax.legend(title='Step', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
        display(df_show)


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                         UI UPDATE HELPERS                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _update_cycle_dropdowns():
    if pressure_df.empty:
        for d in [cycle_dropdown, steady_cycle_dropdown]:
            d.options, d.value = ['All'], 'All'
        return
    cycles  = sorted(pressure_df[pressure_df['cycle'] > 0]['cycle'].unique().astype(int))
    options = ['All'] + cycles
    for d, cb in [(cycle_dropdown, _on_cycle_change),
                  (steady_cycle_dropdown, plot_steady_state_patch)]:
        try:
            d.unobserve_all()
        except Exception:
            pass
        d.options = options
        d.value   = 'All'
        d.observe(cb, names='value')


def _update_step_selectors():
    opts = ['All'] + (step_df['Step'].dropna().unique().tolist() if not step_df.empty else [])
    for sel in [max_step_selector, steady_step_selector]:
        sel.options = opts
        sel.value   = ['All']


def _update_header_tab():
    with header_footer_data_output:
        clear_output(wait=True)
        if not header_info:
            print("No header data available.")
            return
        print(f"Reactor : {header_info.get('reactorName', '')}")
        print(f"GUID    : {header_info.get('GUID', '')}")
        print(f"User    : {header_info.get('username', '')}")
        print(f"Details : {header_info.get('experimentalDetails', '')}")
        print(f"Start   : {header_info.get('startTime', '')}")
        vs = header_info.get('valveSequence')
        if vs:
            print("\nValve Sequence:")
            for row in vs:
                print("  " + "  ".join(f"{x:>6}" for x in row))


def _update_missing_tab():
    with missing_pressure_output:
        clear_output(wait=True)
        if pressure_df.empty:
            print("No data loaded.")
        else:
            n = int(pressure_df['pressure'].isna().sum())
            print(f"Missing pressure packets: {n:,}")


def _update_all_tabs(change=None):
    _update_cycle_dropdowns()
    _update_step_selectors()
    with pressure_output:
        clear_output(wait=True)
        plot_pressure(None)
    update_max_pressure()
    update_steady_state_scatter()
    plot_steady_state_patch()
    update_pressure_overlay()
    _update_header_tab()
    _update_missing_tab()


# ── Per-widget callbacks ──────────────────────────────────────────────────────
def _on_cycle_change(change):
    val = change['new']
    pressure_loading.value = f"Plotting cycle {val}\u2026"
    with pressure_output:
        clear_output(wait=True)
        plot_pressure(None if val == 'All' else val)
    pressure_loading.value = ''


def _on_ss_param_change(change):
    ss['frac']      = ss_frac_w.value
    ss['slope_tol'] = ss_slope_w.value
    ss['amp_tol']   = ss_amp_w.value
    ss['min_pts']   = int(ss_minpts_w.value)
    _compute_steady()
    plot_steady_state_patch()
    update_steady_state_scatter()


print("\u2714 Functions loaded.  Run the next cell to display the GUI.")


✔ Functions loaded.  Run the next cell to display the GUI.


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
#  REACTOR PRESSURE ANALYZER  —  Cell 2: Build & Display GUI
#  Re-run this cell at any time to reset the interface.
# ═══════════════════════════════════════════════════════════════════════════════

# ── File loader ───────────────────────────────────────────────────────────────
file_path_text = widgets.Text(
    value='', placeholder='Paste or browse for a .json file path',
    layout=widgets.Layout(width='500px'),
)
browse_button = widgets.Button(description='Browse\u2026', icon='folder-open',
                               layout=widgets.Layout(width='100px'))
load_button   = widgets.Button(description='Load', icon='upload',
                               layout=widgets.Layout(width='80px'))
load_progress = widgets.IntProgress(value=0, min=0, max=100, bar_style='info',
                                    layout=widgets.Layout(width='270px'))
load_status   = widgets.Label(value='')

browse_button.on_click(browse_file)
load_button.on_click(load_file)

# ── Cycle navigation ──────────────────────────────────────────────────────────
cycle_dropdown = widgets.Dropdown(
    description='Cycle:', options=['All'], value='All',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='210px'),
)
prev_btn = widgets.Button(description='\u25c4 Prev', layout=widgets.Layout(width='85px'))
next_btn = widgets.Button(description='Next \u25ba', layout=widgets.Layout(width='85px'))

def _nav_cycle(delta, b=None):
    if pressure_df.empty:
        return
    opts = list(cycle_dropdown.options)
    cur  = cycle_dropdown.value
    idx  = opts.index(cur) if cur in opts else 0
    cycle_dropdown.value = opts[(idx + delta) % len(opts)]

prev_btn.on_click(lambda b: _nav_cycle(-1))
next_btn.on_click(lambda b: _nav_cycle(+1))

steady_cycle_dropdown = widgets.Dropdown(
    description='Cycle:', options=['All'], value='All',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='210px'),
)

# ── Step selectors ────────────────────────────────────────────────────────────
max_step_selector = widgets.SelectMultiple(
    options=['All'], value=['All'], description='Step(s):',
    rows=9, style={'description_width': 'initial'},
    layout=widgets.Layout(width='270px'),
)
steady_step_selector = widgets.SelectMultiple(
    options=['All'], value=['All'], description='Step(s):',
    rows=9, style={'description_width': 'initial'},
    layout=widgets.Layout(width='270px'),
)

# ── Output areas ──────────────────────────────────────────────────────────────
pressure_output           = widgets.Output()
pressure_overlay_output   = widgets.Output()
max_pressure_output       = widgets.Output()
steady_state_patch_output = widgets.Output()
steady_state_output       = widgets.Output()
header_footer_data_output = widgets.Output()
missing_pressure_output   = widgets.Output()
pressure_loading          = widgets.Label(value='')

# ── Steady-state parameter widgets ────────────────────────────────────────────
ss_frac_w   = widgets.BoundedFloatText(value=ss['frac'],      min=0.01, max=1.0, step=0.01,
                                        description='Steady Fraction:',
                                        style={'description_width': 'initial'},
                                        layout=widgets.Layout(width='225px'))
ss_slope_w  = widgets.BoundedFloatText(value=ss['slope_tol'], min=0.0,           step=0.1,
                                        description='Slope Tol (mTorr/s):',
                                        style={'description_width': 'initial'},
                                        layout=widgets.Layout(width='225px'))
ss_amp_w    = widgets.BoundedFloatText(value=ss['amp_tol'],   min=0.0,           step=1.0,
                                        description='Amp Tol (mTorr):',
                                        style={'description_width': 'initial'},
                                        layout=widgets.Layout(width='225px'))
ss_minpts_w = widgets.BoundedIntText(  value=ss['min_pts'],   min=2,             step=1,
                                        description='Min Points:',
                                        style={'description_width': 'initial'},
                                        layout=widgets.Layout(width='225px'))

# ── Wire observers ────────────────────────────────────────────────────────────
cycle_dropdown.observe(_on_cycle_change, names='value')
max_step_selector.observe(update_max_pressure, names='value')
steady_step_selector.observe(update_steady_state_scatter, names='value')
steady_cycle_dropdown.observe(plot_steady_state_patch, names='value')
for _w in [ss_frac_w, ss_slope_w, ss_amp_w, ss_minpts_w]:
    _w.observe(_on_ss_param_change, names='value')

# ── Tab layout ────────────────────────────────────────────────────────────────
overlay_accordion = widgets.Accordion(children=[pressure_overlay_output])
overlay_accordion.set_title(0, 'Overlay of All Cycles')

misc_html = widgets.HTML("""
<b><u>Steady-State Parameters</u></b>
<ol>
  <li><b>Steady Fraction</b>: Fraction of final points in a step used for analysis (e.g. 0.2 = last 20%).</li>
  <li><b>Slope Tolerance (mTorr/s)</b>: Max allowed fitted slope for a step to be "steady" (dP/dt \u2248 0).</li>
  <li><b>Amplitude Tolerance (mTorr)</b>: Max allowed peak-to-peak fluctuation within the steady window.</li>
  <li><b>Minimum Points</b>: Minimum number of pressure readings required in the steady window.</li>
</ol>
<br>
Use <b>Shift</b> or <b>Ctrl/Cmd + click</b> to select multiple steps in the step selector lists.
<br><br>
<b>File Loading:</b> Click <b>Browse</b> to pick the file (local kernel only) or paste the full path, 
then click <b>Load</b>. The parser is streaming \u2014 only pressure and step records are retained in memory,
so large files (100 MB+) load quickly without stalling the kernel.
""")

tab = widgets.Tab(children=[
    widgets.VBox([
        widgets.HTML('<b>Pressure vs Time</b>, background color-coded by step.'),
        widgets.HBox([cycle_dropdown, prev_btn, next_btn]),
        pressure_loading,
        pressure_output,
        overlay_accordion,
    ]),
    widgets.VBox([
        widgets.HTML('<b>Max Pressure</b> per step per cycle.'),
        max_step_selector,
        max_pressure_output,
    ]),
    widgets.VBox([
        widgets.HTML('<b>Steady-State Classification</b> \u2014 green\u00a0=\u00a0steady, red\u00a0=\u00a0not steady.'),
        steady_cycle_dropdown,
        steady_state_patch_output,
        widgets.HBox([ss_frac_w, ss_slope_w, ss_amp_w, ss_minpts_w]),
        steady_step_selector,
        steady_state_output,
    ]),
    widgets.VBox([
        widgets.HTML('<b>Experiment / Header Data</b>'),
        header_footer_data_output,
    ]),
    widgets.VBox([
        widgets.HTML('<b>Missing Pressure Data</b>'),
        missing_pressure_output,
    ]),
    misc_html,
])

for _i, _title in enumerate(['Pressure Plot', 'Max Plot', 'Steady-State',
                              'Experiment Info', 'Missing Data', 'Info']):
    tab.set_title(_i, _title)

# ── Render ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([file_path_text, browse_button, load_button]),
    widgets.HBox([load_progress, load_status]),
]))
display(tab)


In [ ]:
# header_info


In [ ]:
# pressure_df  — columns: time_ms, pressure, cycle


In [ ]:
# step_df